# Thermal Refuge Equity — Curico & Valdivia (2016 vs 2026)

**Question:** within each city, how much of the population lives in the areas with the worst relative access to thermal relief (vegetation + cooling), and how has that changed over a decade?

This notebook builds a **Cooling Index** per H3 hexagon (resolution 9, ~0.09 km² locally) from Landsat NDVI/LST, standardized within each city-year, and cross-references it with modeled population (WorldPop) to compute an equity metric: the share of the population living in the worst-off quartile of hexagons.

**Why standardize within city-year, not across cities:** Curico and Valdivia sit in different climate zones. Comparing raw NDVI/LST values between them would conflate climate with urban design. Z-scoring within each city-year isolates *relative position inside that city's own context*, which is the quantity that matters for an equity question.

**Pipeline**
1. H3 grid over each city's built-up extent (GHSL Urban Centre Database)
2. NDVI + Land Surface Temperature composites (Landsat 8/9, Collection 2 Level 2)
3. Zonal aggregation to hexagons + Cooling Index (`z(NDVI) - z(LST)`)
4. Population per hexagon (WorldPop)
5. Equity metric: % of population in the worst thermal quartile

**Known limitations** (see full write-up in `docs/DEVLOG.md`):
- WorldPop 2026 is a model projection, not a census measurement.
- WorldPop population totals here use a *constrained* settlement model — expect them to run ~15% below unconstrained sources (e.g. GHSL) for the same area.
- Landsat thermal-band coverage should be screened per city before committing to it (see README) — it can vary independently of optical-band quality.


In [ ]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Point, Polygon, mapping

ee.Initialize()


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for parent in [start, *start.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"No .git directory found upward from {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs" / "qgis_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

H3_RESOLUTION = 9
MIN_VALID_FRACTION = 0.5
NODATA_VALUE = -9999.0
SEASON_START, SEASON_END = "-01-01", "-03-31"  # austral summer, Jan-Mar

CITY_CENTROIDS = {
    "curico": Point(-71.2394, -34.9828),
    "valdivia": Point(-73.2459, -39.8142),
}

CITY_YEARS = [
    ("curico", 2016),
    ("curico", 2026),
    ("valdivia", 2016),
    ("valdivia", 2026),
]


## Component 1 — H3 grid over each city's built-up extent

City extents come from the **GHSL Urban Centre Database (R2024A)**, a fixed-boundary delineation (epoch 2025) covering built-up settlement, not administrative limits.

Cities are matched by **point-in-polygon containment** using a known city-center coordinate, not by string-matching a name field. UCDB attribute naming has changed across releases and is subject to encoding issues with accented characters — geometric containment is more robust and doesn't depend on getting a column name right.

Hexagons are kept intact at the AOI border rather than clipped to the UCDB polygon. Clipping would produce variable-area edge hexagons, which would bias any later per-area normalization (e.g. population density).


In [ ]:
ucdb_path = (
    DATA_DIR / "ghsl"
    / "GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A_V1_2"
    / "GHS_UCDB_REGION_LATIN_AMERICA_AND_THE_CARIBBEAN_R2024A.gpkg"
)
UCDB_LAYER = "GHSL_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A"

ucdb = gpd.read_file(ucdb_path, layer=UCDB_LAYER).to_crs(epsg=4326)

city_polygons: dict[str, Polygon] = {}
for city, point in CITY_CENTROIDS.items():
    match = ucdb[ucdb.contains(point)]
    if len(match) != 1:
        raise ValueError(f"{city}: expected 1 matching polygon, found {len(match)}")
    city_polygons[city] = match.geometry.iloc[0]
    print(city, "->", match["GC_UCN_MAI_2025"].values[0], "|", match["GC_CNT_UNN_2025"].values[0])


In [ ]:
def hexagons_for_polygon(polygon: Polygon, resolution: int) -> gpd.GeoDataFrame:
    """H3-polyfill a polygon; keep whole hexagons, no geometric clipping."""
    h3_shape = h3.geo_to_h3shape(mapping(polygon))
    cell_ids = h3.polygon_to_cells(h3_shape, res=resolution)
    records = [
        {"hex_id": cell_id, "geometry": Polygon(
            [(lon, lat) for lat, lon in h3.cell_to_boundary(cell_id)]
        )}
        for cell_id in cell_ids
    ]
    return gpd.GeoDataFrame(records, crs="EPSG:4326")


hex_frames = []
for city, polygon in city_polygons.items():
    hex_gdf = hexagons_for_polygon(polygon, H3_RESOLUTION)
    hex_gdf["city"] = city
    hex_frames.append(hex_gdf)
    print(city, "-", len(hex_gdf), "hexagons")

hex_grid = gpd.GeoDataFrame(pd.concat(hex_frames, ignore_index=True), crs="EPSG:4326")


In [ ]:
# H3 is not strictly equal-area; local cell area varies with latitude.
# Reproject and export — this is the AOI reference for all downstream steps.
hex_grid_utm = hex_grid.to_crs(epsg=32718)
hex_grid_utm["area_km2"] = hex_grid_utm.geometry.area / 1e6
print(hex_grid_utm.groupby("city")["area_km2"].agg(["count", "mean", "std"]))

hex_grid_path = OUTPUT_DIR / "hex_grid_res9.gpkg"
hex_grid_utm.to_file(hex_grid_path, driver="GPKG")
print("Exported:", hex_grid_path)


## Component 2 — NDVI and Land Surface Temperature per city-year

Landsat 8/9 Collection 2 Level 2, `ST_B10` for surface temperature, austral-summer window (Jan-Mar). Landsat 8 only for 2016 (L9 wasn't launched yet); L8+L9 combined for 2026.

AOI is the H3 grid's bounding box per city, buffered 500 m, to avoid edge hexagons losing pixels.

Composites are pixel-wise median over the season, masked with standard `QA_PIXEL` cloud/shadow/cirrus bits. Fully-masked pixels are explicitly filled with a sentinel nodata value before export — Earth Engine's export does not preserve masked-pixel status as readable nodata by default, and downstream zonal statistics need an explicit value to distinguish "no data" from "valid measurement."


In [ ]:
def city_aoi(city: str) -> ee.Geometry:
    hexes = hex_grid[hex_grid["city"] == city]
    bbox = hexes.total_bounds  # minx, miny, maxx, maxy, EPSG:4326
    return ee.Geometry.Rectangle(list(bbox)).buffer(500)


def mask_qa_pixel(image: ee.Image) -> ee.Image:
    qa = image.select("QA_PIXEL")
    dilated_cloud, cirrus, cloud, cloud_shadow = 1 << 1, 1 << 2, 1 << 3, 1 << 4
    mask = (
        qa.bitwiseAnd(dilated_cloud).eq(0)
        .And(qa.bitwiseAnd(cirrus).eq(0))
        .And(qa.bitwiseAnd(cloud).eq(0))
        .And(qa.bitwiseAnd(cloud_shadow).eq(0))
    )
    return image.updateMask(mask)


def add_ndvi_lst(image: ee.Image) -> ee.Image:
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    lst_celsius = (
        image.select("ST_B10")
        .multiply(0.00341802).add(149.0).subtract(273.15)
        .rename("LST")
    )
    return image.addBands([ndvi, lst_celsius])


def build_composite(city: str, year: int) -> ee.Image:
    aoi = city_aoi(city)
    date_start, date_end = f"{year}{SEASON_START}", f"{year}{SEASON_END}"

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
    collection = l8
    if year != 2016:
        l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(date_start, date_end)
        collection = l8.merge(l9)

    processed = collection.map(mask_qa_pixel).map(add_ndvi_lst)
    composite = processed.select(["NDVI", "LST"]).median().clip(aoi)
    return composite.unmask(NODATA_VALUE)


In [ ]:
raster_paths = {
    (city, year): OUTPUT_DIR / f"{city}_{year}_ndvi_lst.tif"
    for (city, year) in CITY_YEARS
}


def export_composite(city: str, year: int) -> None:
    composite = build_composite(city, year)
    output_path = raster_paths[(city, year)]
    if output_path.exists():
        output_path.unlink()

    geemap.ee_export_image(
        composite, filename=str(output_path), scale=30, region=city_aoi(city), file_per_band=False,
    )

    if not output_path.exists():
        raise RuntimeError(f"{output_path.name}: export did not produce a file")

    with rasterio.open(output_path) as src:
        n_sentinel = (src.read(1) == NODATA_VALUE).sum()
    print(f"{output_path.name}: {output_path.stat().st_size / 1e6:.2f} MB, {n_sentinel} nodata px")


for city, year in CITY_YEARS:
    export_composite(city, year)


## Component 3 — Zonal aggregation and the Cooling Index

NDVI and LST are aggregated to each hexagon as a **coverage-weighted mean**, with a minimum-valid-pixel threshold (50%) applied **independently per band**. NDVI and the thermal band do not fail in the same pixels — the ST retrieval algorithm has its own, stricter internal masking — so a hexagon can have a perfectly good NDVI mean and no valid LST mean, or vice versa. Applying one band's coverage fraction as a proxy for the other silently corrupts the result.

The Cooling Index is `z(NDVI) - z(LST)`, standardized within each `(city, year)` group.


In [ ]:
def zonal_aggregate(city: str, year: int, hex_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    raster_path = raster_paths[(city, year)]
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

    hex_local = hex_gdf[hex_gdf["city"] == city].to_crs(raster_crs).reset_index(drop=True)
    stats_ndvi = zonal_stats(hex_local, str(raster_path), band=1, stats=["mean", "count"], nodata=NODATA_VALUE)
    stats_lst = zonal_stats(hex_local, str(raster_path), band=2, stats=["mean", "count"], nodata=NODATA_VALUE)

    records = []
    for i, row in hex_local.iterrows():
        n_theoretical = row.geometry.area / pixel_area_m2
        vf_ndvi = (stats_ndvi[i]["count"] or 0) / n_theoretical if n_theoretical > 0 else 0.0
        vf_lst = (stats_lst[i]["count"] or 0) / n_theoretical if n_theoretical > 0 else 0.0

        records.append({
            "hex_id": row["hex_id"],
            "city": city,
            "year": year,
            "ndvi_mean": stats_ndvi[i]["mean"] if vf_ndvi >= MIN_VALID_FRACTION else np.nan,
            "lst_mean": stats_lst[i]["mean"] if vf_lst >= MIN_VALID_FRACTION else np.nan,
            "valid_fraction_ndvi": round(vf_ndvi, 3),
            "valid_fraction_lst": round(vf_lst, 3),
        })
    return pd.DataFrame(records)


zonal_frames = [zonal_aggregate(city, year, hex_grid) for city, year in CITY_YEARS]
metrics = pd.concat(zonal_frames, ignore_index=True)

print(metrics.groupby(["city", "year"])[["valid_fraction_ndvi", "valid_fraction_lst"]].mean())
print("Hexagons dropped — NDVI:", metrics["ndvi_mean"].isna().sum(),
      "| LST:", metrics["lst_mean"].isna().sum())


In [ ]:
def zscore(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / series.std()


metrics["z_ndvi"] = metrics.groupby(["city", "year"])["ndvi_mean"].transform(zscore)
metrics["z_lst"] = metrics.groupby(["city", "year"])["lst_mean"].transform(zscore)
metrics["cooling_index"] = metrics["z_ndvi"] - metrics["z_lst"]

# sanity check: mean ~0, std == 1 within every city-year group
print(metrics.groupby(["city", "year"])[["z_ndvi", "z_lst"]].agg(["mean", "std"]))


In [ ]:
hex_metrics = hex_grid_utm.drop(columns="area_km2").merge(metrics, on=["hex_id", "city"], how="left")
hex_metrics_path = OUTPUT_DIR / "hex_cooling_index.gpkg"
hex_metrics.to_file(hex_metrics_path, driver="GPKG")
print("Exported:", hex_metrics_path)


## Component 4 — Population per hexagon

Population comes from a **constrained** WorldPop model (`sat-io` community catalog on Earth Engine), which allocates population only to built-settlement pixels — not the official Google-curated `WorldPop/GP/100m/pop` collection, which stops at 2021 and has no 2026 epoch.

Population is aggregated as an **area-weighted sum**, not a mean — it's a count, not an intensity.

Note: `year` in this collection is stored as a **string**, not an integer; filtering with an int silently returns an empty collection.


In [ ]:
worldpop = ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")


def population_for_city_year(city: str, year: int, hex_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    hex_local = hex_gdf[hex_gdf["city"] == city][["hex_id", "geometry"]].to_crs(epsg=4326)
    hex_fc = geemap.geopandas_to_ee(hex_local)

    pop_image = (
        worldpop
        .filter(ee.Filter.eq("country_code", "CHL"))
        .filter(ee.Filter.eq("year", str(year)))
        .first()
    )
    result_fc = pop_image.reduceRegions(collection=hex_fc, reducer=ee.Reducer.sum(), scale=100)
    features = result_fc.getInfo()["features"]

    return pd.DataFrame([
        {"hex_id": f["properties"]["hex_id"], "city": city, "year": year,
         "population": f["properties"].get("sum", 0.0)}
        for f in features
    ])


pop_frames = [population_for_city_year(city, year, hex_grid) for city, year in CITY_YEARS]
population_df = pd.concat(pop_frames, ignore_index=True)
print(population_df.groupby(["city", "year"])["population"].sum())


In [ ]:
hex_metrics = gpd.read_file(hex_metrics_path)
hex_metrics = hex_metrics.merge(population_df, on=["hex_id", "city", "year"], how="left")
print("Hexagons missing population:", hex_metrics["population"].isna().sum())

hex_metrics.to_file(hex_metrics_path, driver="GPKG")
print("Updated:", hex_metrics_path)


## Component 5 — Equity metric

Hexagons are split into quartiles of `cooling_index`, **within each city-year**. Quartile 1 = the 25% of hexagons with the worst relative cooling index in that city that year.

**Reference point:** if population were evenly spread across thermal quartiles, quartile 1 would hold ~25% of the population by construction. Deviation above 25% signals population concentrated in thermally disadvantaged areas; below 25% signals a protective pattern.


In [ ]:
def assign_quartile(series: pd.Series) -> pd.Series:
    return pd.qcut(series, q=4, labels=[1, 2, 3, 4])


hex_metrics["cooling_quartile"] = hex_metrics.groupby(
    ["city", "year"], observed=True
)["cooling_index"].transform(assign_quartile)

hex_metrics.to_file(hex_metrics_path, driver="GPKG")
print("Updated with quartiles:", hex_metrics_path)


In [ ]:
def population_in_worst_quartile(df: pd.DataFrame) -> pd.Series:
    total = df["population"].sum()
    worst = df.loc[df["cooling_quartile"] == 1, "population"].sum()
    return pd.Series({
        "population_total": total,
        "population_worst_quartile": worst,
        "pct_population_worst_quartile": 100 * worst / total,
    })


equity = hex_metrics.groupby(["city", "year"]).apply(population_in_worst_quartile, include_groups=False)
equity_wide = equity["pct_population_worst_quartile"].unstack("year")
equity_wide["delta_2016_2026"] = equity_wide[2026] - equity_wide[2016]

print(equity_wide)

equity.reset_index().to_csv(OUTPUT_DIR / "thermal_equity.csv", index=False)


## Interactive map — exploratory HTML output

Static summary numbers hide where exactly the worst-off quartile sits inside each city.
This builds two standalone Folium maps (one per city, each centered and zoomed to its own
extent — the two cities are too far apart geographically to share a single view) with
togglable layers: Cooling Index for each year, the year-over-year change, and a
"population at risk" layer isolating the worst quartile weighted by population.

Output: `docs/index.html` (embeds both maps side by side) plus `docs/map_curico.html` and
`docs/map_valdivia.html` (the two standalone Folium maps it embeds via iframe).


In [ ]:
import folium
import branca.colormap as cm

DOCS_DIR = REPO_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

ACCENT_COOL = "#0E6B62"
ACCENT_WARM = "#B23A2E"
CITY_LABELS = {"curico": "Curico", "valdivia": "Valdivia"}

CAVEAT_HTML_TEMPLATE = """
<div style="position: fixed; bottom: 16px; left: 16px; z-index: 9999;
            background: white; padding: 12px 16px; max-width: 300px;
            font-family: 'Public Sans', Helvetica, Arial, sans-serif;
            font-size: 12.5px; line-height: 1.5; color: #333;
            border-radius: 4px; box-shadow: 0 1px 6px rgba(0,0,0,0.18);">
  <b style="font-size: 13.5px;">Cooling Index — {city_label}</b><br>
  Standardized within this city and year (z-NDVI minus z-LST). Values compare
  a hexagon's relative position to the rest of this city, not absolute
  temperature between cities.
</div>
"""


In [ ]:
def build_city_map(city_key: str, hex_metrics_wgs84: gpd.GeoDataFrame) -> str:
    """Build a standalone Folium map for one city and save it to docs/."""
    city_label = CITY_LABELS[city_key]
    city_gdf = hex_metrics_wgs84[hex_metrics_wgs84["city"] == city_key].copy()

    bounds = city_gdf.total_bounds
    center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

    m = folium.Map(location=center, zoom_start=13, tiles=None, control_scale=True)
    folium.TileLayer("cartodbvoyager", name="cartodbvoyager", control=False).add_to(m)

    ci_min, ci_max = hex_metrics_wgs84["cooling_index"].min(), hex_metrics_wgs84["cooling_index"].max()
    cooling_cmap = cm.LinearColormap(
        colors=[ACCENT_WARM, "#F4E9D8", ACCENT_COOL], vmin=ci_min, vmax=ci_max,
        caption=f"Cooling Index ({city_label})",
    )

    delta_gdf = city_gdf.pivot(index="hex_id", columns="year", values="cooling_index").reset_index()
    delta_gdf["delta"] = delta_gdf[2026] - delta_gdf[2016]
    geom_lookup = city_gdf.drop_duplicates("hex_id").set_index("hex_id")["geometry"]
    delta_gdf = gpd.GeoDataFrame(
        delta_gdf.merge(geom_lookup, left_on="hex_id", right_index=True),
        geometry="geometry", crs="EPSG:4326",
    )
    delta_max_abs = max(abs(delta_gdf["delta"].min()), abs(delta_gdf["delta"].max()), 0.01)
    delta_cmap = cm.LinearColormap(
        colors=[ACCENT_WARM, "#F4E9D8", ACCENT_COOL], vmin=-delta_max_abs, vmax=delta_max_abs,
        caption=f"Δ Cooling Index, 2026 minus 2016 ({city_label})",
    )

    for year in [2016, 2026]:
        year_gdf = city_gdf[city_gdf["year"] == year]
        fg = folium.FeatureGroup(name=f"Cooling Index {year}", show=False)
        for _, row in year_gdf.iterrows():
            has_ci = row["cooling_index"] == row["cooling_index"]
            fill = cooling_cmap(row["cooling_index"]) if has_ci else "#CCCCCC"
            tooltip = (
                (f"Cooling Index: {row['cooling_index']:.2f}" if has_ci else "No data")
                + (f"<br>Population (est.): {row['population']:.0f}" if row["population"] == row["population"] else "")
                + (f"<br>Quartile: {int(row['cooling_quartile'])} (1 = worst)"
                   if row["cooling_quartile"] == row["cooling_quartile"] else "")
            )
            folium.GeoJson(
                row["geometry"].__geo_interface__,
                style_function=lambda _f, fill=fill: {
                    "fillColor": fill, "fillOpacity": 0.5, "color": "#FAF9F6", "weight": 0.4,
                },
                tooltip=folium.Tooltip(tooltip),
            ).add_to(fg)
        fg.add_to(m)

    worst_gdf = city_gdf[(city_gdf["cooling_quartile"] == 1) & (city_gdf["year"] == 2026)].copy()
    pop_min, pop_max = worst_gdf["population"].min(), worst_gdf["population"].max()
    pop_range = max(pop_max - pop_min, 1e-6)

    fg_risk = folium.FeatureGroup(name="Population at risk (worst quartile, 2026)", show=False)
    for _, row in worst_gdf.iterrows():
        pop = row["population"] if row["population"] == row["population"] else 0.0
        opacity = 0.15 + 0.75 * ((pop - pop_min) / pop_range)
        tooltip = f"Population (est.): {pop:.0f}<br>Cooling Index: {row['cooling_index']:.2f} (worst quartile)"
        folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda _f, opacity=opacity: {
                "fillColor": ACCENT_WARM, "fillOpacity": opacity, "color": "#FAF9F6", "weight": 0.4,
            },
            tooltip=folium.Tooltip(tooltip),
        ).add_to(fg_risk)
    fg_risk.add_to(m)

    fg_delta = folium.FeatureGroup(name="Δ Cooling Index (2026 − 2016)", show=False)
    for _, row in delta_gdf.iterrows():
        has_delta = row["delta"] == row["delta"]
        fill = delta_cmap(row["delta"]) if has_delta else "#CCCCCC"
        tooltip = f"Δ Cooling Index: {row['delta']:+.2f}" if has_delta else "No data"
        folium.GeoJson(
            row["geometry"].__geo_interface__,
            style_function=lambda _f, fill=fill: {
                "fillColor": fill, "fillOpacity": 0.5, "color": "#FAF9F6", "weight": 0.4,
            },
            tooltip=folium.Tooltip(tooltip),
        ).add_to(fg_delta)
    fg_delta.add_to(m)

    cooling_cmap.add_to(m)
    delta_cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(folium.Element(CAVEAT_HTML_TEMPLATE.format(city_label=city_label)))

    out_path = DOCS_DIR / f"map_{city_key}.html"
    m.save(str(out_path))
    return str(out_path)


In [ ]:
hex_metrics_wgs84 = hex_metrics.to_crs(epsg=4326)

for city_key in ["curico", "valdivia"]:
    path = build_city_map(city_key, hex_metrics_wgs84)
    print("saved:", path)


The `docs/index.html` wrapper page (embeds both maps side by side, adds the project
header and the plain-language explanation of the method) is a static file, not
regenerated from data — it only needs to change if the narrative text changes, not when
the underlying analysis reruns. It's committed directly rather than generated here.


## Results summary

| City | 2016 | 2026 | Δ |
|---|---|---|---|
| Curico | 26.2% | 28.2% | +1.9 |
| Valdivia | 42.4% | 42.0% | -0.5 |

Curico sits close to the neutral 25% baseline, with a slight upward drift. Valdivia holds nearly double the population share one would expect by chance, stable across the decade — visually confirmed against satellite imagery as a contiguous cluster over the dense urban core, not an artifact of a few high-population outlier hexagons.

This does not claim Valdivia is hotter than Curico in absolute terms — the within-city standardization specifically prevents that comparison. It claims that exposure to relative thermal disadvantage is far more unequally distributed among Valdivia's own residents than among Curico's.

Full methodology notes, incident log, and rejected alternatives are in `docs/DEVLOG.md`.
